In [ ]:
!pip install yfinance pandas boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.8 MB/s eta 0:00:00


In [ ]:
import yfinance as yf
import pandas as pd

# ==========================================
# 1. PARAMETER DATA
# ==========================================
ticker = "MEDC.JK"
periode = "5y" # Mengambil data 5 tahun terakhir untuk menangkap tren makro

print(f"Mulai mengunduh data saham {ticker} dari Yahoo Finance API...")

# ==========================================
# 2. PROSES UNDUH DATA
# ==========================================
data = yf.download(ticker, period=periode)

# Mengecek apakah data berhasil ditarik
if data.empty:
    print("Gagal menarik data. Cek koneksi internet atau simbol ticker.")
else:
    print("Data berhasil diunduh!")

    # ==========================================
    # 3. PEMBERSIHAN DATA (CLEANING)
    # ==========================================
    # Menghapus baris yang memiliki nilai kosong
    data_clean = data.dropna()

    # Mengembalikan 'Date' dari index menjadi kolom biasa
    data_clean = data_clean.reset_index()

    # Memilih kolom esensial yang umum digunakan untuk forecasting
    kolom_utama = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    data_final = data_clean[kolom_utama]

    # ==========================================
    # 4. INSPEKSI ANALITIK
    # ==========================================
    print("\n--- 5 Baris Pertama Data ---")
    print(data_final.head())

    print("\n--- Info Struktur Data ---")
    print(data_final.info())

    print("\n--- Ringkasan Statistik ---")
    print(data_final.describe())

    # ==========================================
    # 5. SIMPAN KE LOKAL (UNTUK UJI COBA)
    # ==========================================
    nama_file = "medc.csv"
    data_final.to_csv(nama_file, index=False)
    print(f"\nData berhasil disimpan secara lokal dengan nama: {nama_file}")

Mulai mengunduh data saham MEDC.JK dari Yahoo Finance API...


/tmp/ipykernel_2353/748794045.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=periode)
[*********************100%***********************]  1 of 1 completed

Data berhasil diunduh!

--- 5 Baris Pertama Data ---
Price        Date        Open        High         Low       Close    Volume
Ticker                MEDC.JK     MEDC.JK     MEDC.JK     MEDC.JK   MEDC.JK
0      2021-04-13  483.732533  483.732533  471.113424  475.319794  17371600
1      2021-04-14  483.732534  487.938904  475.319794  487.938904  14964900
2      2021-04-15  500.558013  504.764383  483.732534  487.938904  22801200
3      2021-04-16  487.938945  496.351685  483.732574  483.732574  16677600
4      2021-04-19  483.732533  487.938903  475.319794  475.319794  16503300

--- Info Struktur Data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   (Date, )           1200 non-null   datetime64[ns]
 1   (Open, MEDC.JK)    1200 non-null   float64       
 2   (High, MEDC.JK)    1200 non-null   float64       
 3   (

In [ ]:
data_final

Price,Date,Open,High,Low,Close,Volume
Ticker,,MEDC.JK,MEDC.JK,MEDC.JK,MEDC.JK,MEDC.JK
0,2021-04-13,483.732533,483.732533,471.113424,475.319794,17371600
1,2021-04-14,483.732534,487.938904,475.319794,487.938904,14964900
2,2021-04-15,500.558013,504.764383,483.732534,487.938904,22801200
3,2021-04-16,487.938945,496.351685,483.732574,483.732574,16677600
4,2021-04-19,483.732533,487.938903,475.319794,475.319794,16503300
...,...,...,...,...,...,...
1195,2026-04-07,1600.000000,1660.000000,1600.000000,1630.000000,84524400
1196,2026-04-08,1570.000000,1570.000000,1475.000000,1555.000000,259045700
1197,2026-04-09,1580.000000,1615.000000,1545.000000,1550.000000,129946100


In [ ]:
data_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   (Date, )           1200 non-null   datetime64[ns]
 1   (Open, MEDC.JK)    1200 non-null   float64       
 2   (High, MEDC.JK)    1200 non-null   float64       
 3   (Low, MEDC.JK)     1200 non-null   float64       
 4   (Close, MEDC.JK)   1200 non-null   float64       
 5   (Volume, MEDC.JK)  1200 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 56.4 KB


In [ ]:
# Export Data CSV
data_final.to_csv("energy_data.csv", index=False)

# CONNECT AWS

In [ ]:
!pip install yfinance boto3 pandas matplotlib -q

import yfinance as yf
import pandas as pd
import boto3
from google.colab import userdata
import matplotlib.pyplot as plt

In [ ]:
s3 = boto3.client(
    's3',
    region_name='ap-southeast-1',
    aws_access_key_id=userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=userdata.get('AWS_SECRET_ACCESS_KEY')
)

BUCKET_NAME = "medc-energy-forecasting"

In [ ]:
s3.upload_file("energy_data.csv", BUCKET_NAME, "raw/energy_data.csv")
print("Upload ke S3:/raw/energy_data.csv berhasil!")

Upload ke S3:/raw/energy_data.csv berhasil!


In [ ]:
s3.download_file(BUCKET_NAME, "raw/energy_data.csv", "verify_data.csv")
df_verify = pd.read_csv("verify_data.csv")
print("\n Verifikasi baca dari S3:")
print(df_verify.head())


 Verifikasi baca dari S3:
         Date               Open                High                Low  \
0         NaN            MEDC.JK             MEDC.JK            MEDC.JK   
1  2021-04-13   483.732533412697    483.732533412697  471.1134238454093   
2  2021-04-14  483.7325339481748  487.93890380859375   475.319794227337   
3  2021-04-15  500.5580133898505  504.76438325026936  483.7325339481748   
4  2021-04-16  487.9389446756114  496.35168510105296  483.7325744628906   

                Close    Volume  
0             MEDC.JK   MEDC.JK  
1   475.3197937011719  17371600  
2  487.93890380859375  14964900  
3  487.93890380859375  22801200  
4   483.7325744628906  16677600  
